cloude aı deneme 

In [1]:
"""
YouTube Yorum Etiketleme Scripti
=================================
Kullanılan modeller:
  1. cardiffnlp/twitter-xlm-roberta-base-sentiment  (çok dilli, sosyal medyaya özel)
  2. savasy/bert-base-turkish-sentiment-cased        (Türkçe'ye özel)

Sınıflar: olumlu | olumsuz | öneri/tavsiye | şikayet | nötr

Kurulum:
  pip install transformers torch pandas tqdm sentencepiece protobuf

Kullanım:
  python yorum_etiketleme.py
"""

import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)
from tqdm import tqdm
import re
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0. AYARLAR
# ─────────────────────────────────────────────
INPUT_FILE  = "toplu_yorum_verileri.csv"
OUTPUT_FILE = "etiketlenmis_yorumlar2.csv"
TEXT_COL    = "Yorum Metni"
BATCH_SIZE  = 32          # GPU yoksa 16'ya düşür
MAX_LENGTH  = 128         # token limiti
DEVICE      = 0 if torch.cuda.is_available() else -1  # -1 = CPU

print(f"Cihaz: {'GPU (cuda)' if DEVICE == 0 else 'CPU'}")

# ─────────────────────────────────────────────
# 1. KURAL TABANLI ÖN FİLTRE
#    Öneri ve şikayet sınıfları için kelime listeleri
# ─────────────────────────────────────────────
ONERI_KALIPLARI = [
    r"\böneri\b", r"\btavsiye\b", r"\böneririm\b", r"\btavsiye ederim\b",
    r"\bşunu yaps", r"\bkeşke\b", r"\bolsa iyi olur\b", r"\byapılabilir\b",
    r"\byaparsanız\b", r"\bekliyoruz\b", r"\bekleriz\b", r"\bgelsin\b",
    r"\byapın\b", r"\bekleyin\b", r"\bgetirin\b", r"\byayınlayın\b",
    r"\büretebilir", r"\bdeneyebilirsiniz\b", r"\bdenemeli\b",
]

SIKAYET_KALIPLARI = [
    r"\bşikayet\b", r"\brezalet\b", r"\bkabul edilemez\b", r"\bhaksızlık\b",
    r"\biğrenç\b", r"\btiksindirici\b", r"\bçok kötü\b", r"\bçok berbat\b",
    r"\batın\b", r"\bgönderin\b", r"\bistemiyoruz\b", r"\bartık izlemiyorum\b",
    r"\bunfollow\b", r"\baboneliği iptal\b", r"\bdefolun\b", r"\bsaçmalık\b",
    r"\bhayal kırıklığı\b", r"\bbezginlik\b", r"\bçektirdiniz\b",
    r"\bbıktım\b", r"\busandım\b", r"\byeter artık\b",
]

ONERI_RE  = re.compile("|".join(ONERI_KALIPLARI),  re.IGNORECASE)
SIKAYET_RE = re.compile("|".join(SIKAYET_KALIPLARI), re.IGNORECASE)


def kural_tabanli_etiket(metin: str):
    """
    Önce kural tabanlı kontrol yap.
    Eşleşme varsa (etiket, confidence) döndür.
    Yoksa None döndür → model devralır.
    """
    if not isinstance(metin, str) or metin.strip() == "":
        return "nötr", 1.0

    if SIKAYET_RE.search(metin):
        return "şikayet", 0.85

    if ONERI_RE.search(metin):
        return "öneri/tavsiye", 0.85

    return None  # model karar verecek


# ─────────────────────────────────────────────
# 2. MODEL YÜKLEME
# ─────────────────────────────────────────────
print("\n[1/2] XLM-RoBERTa yükleniyor...")
xlm_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=DEVICE,
    max_length=MAX_LENGTH,
    truncation=True,
)

# XLM-RoBERTa etiket eşlemesi: Negative/Neutral/Positive → Türkçe
XLM_ETIKET_MAP = {
    "Negative": "olumsuz",
    "Neutral":  "nötr",
    "Positive": "olumlu",
}

print("[2/2] BERTurk Sentiment yükleniyor...")
bert_tr_pipe = pipeline(
    "sentiment-analysis",
    model="savasy/bert-base-turkish-sentiment-cased",
    tokenizer="savasy/bert-base-turkish-sentiment-cased",
    device=DEVICE,
    max_length=MAX_LENGTH,
    truncation=True,
)

# BERTurk etiket eşlemesi
BERT_TR_ETIKET_MAP = {
    "positive": "olumlu",
    "negative": "olumsuz",
    # Bu model nötr çıkarmaz, o yüzden ikisini eşleyip confidence'a bakacağız
}

# ─────────────────────────────────────────────
# 3. VERİ OKUMA & TEMİZLEME
# ─────────────────────────────────────────────
print(f"\nVeri okunuyor: {INPUT_FILE}")
df = pd.read_csv('toplu_yorum_verileri.csv', encoding='utf-8-sig')
print(f"Toplam yorum: {len(df)}")

# küçük bir parçası için 
# Boş / NaN yorumları temizle
df[TEXT_COL] = df[TEXT_COL].fillna("").str.strip()
df = df[df[TEXT_COL] != ""].reset_index(drop=True)
print(f"Boş yorumlar çıkarıldıktan sonra: {len(df)}")

# ─────────────────────────────────────────────
# 4. ETİKETLEME FONKSİYONLARI
# ─────────────────────────────────────────────

def batch_etiketle(pipe, metinler: list, etiket_map: dict):
    """Toplu tahmin yap, liste olarak döndür: [(etiket, confidence), ...]"""
    sonuclar = []
    for i in tqdm(range(0, len(metinler), BATCH_SIZE), desc="  batch", leave=False):
        batch = metinler[i : i + BATCH_SIZE]
        try:
            ciktilar = pipe(batch)
        except Exception as e:
            # Hata durumunda nötr döndür
            ciktilar = [{"label": list(etiket_map.keys())[0], "score": 0.5}] * len(batch)
        for c in ciktilar:
            etiket = etiket_map.get(c["label"], c["label"].lower())
            sonuclar.append((etiket, round(c["score"], 4)))
    return sonuclar


def nihai_etiket_belirle(kural, kural_conf,
                          xlm_etiket, xlm_conf,
                          bert_etiket, bert_conf):
    """
    Öncelik sırası:
    1. Kural tabanlı eşleşme (şikayet / öneri) → direkt kullan
    2. İki model aynı fikirde (conf > 0.6) → o etiketi al
    3. XLM daha yüksek confidence → XLM'i al
    4. Düşük confidence (her ikisi < 0.55) → "nötr"
    """
    # 1. Kural eşleşmesi
    if kural is not None:
        return kural, kural_conf, "kural"

    # 2. İki model aynı fikir
    if xlm_etiket == bert_etiket and xlm_conf > 0.6 and bert_conf > 0.6:
        return xlm_etiket, round((xlm_conf + bert_conf) / 2, 4), "model_uzlaşı"

    # 3. Düşük güven → nötr
    if xlm_conf < 0.55 and bert_conf < 0.55:
        return "nötr", round(max(xlm_conf, bert_conf), 4), "düşük_güven"

    # 4. Yüksek güvenli modeli seç
    if xlm_conf >= bert_conf:
        return xlm_etiket, xlm_conf, "xlm_roberta"
    else:
        return bert_etiket, bert_conf, "berturk"


# ─────────────────────────────────────────────
# 5. ANA ETİKETLEME DÖNGÜSÜ
# ─────────────────────────────────────────────
metinler = df[TEXT_COL].tolist()

# 5a. Kural tabanlı
print("\nKural tabanlı ön filtre uygulanıyor...")
kural_sonuclari = [kural_tabanli_etiket(m) for m in metinler]
kural_etiketi  = [k[0] if k else None for k in kural_sonuclari]
kural_conf     = [k[1] if k else None for k in kural_sonuclari]

# 5b. XLM-RoBERTa
print("\nXLM-RoBERTa ile etiketleniyor...")
xlm_sonuclari = batch_etiketle(xlm_pipe, metinler, XLM_ETIKET_MAP)
xlm_etiket    = [s[0] for s in xlm_sonuclari]
xlm_conf      = [s[1] for s in xlm_sonuclari]

# 5c. BERTurk
print("\nBERTurk ile etiketleniyor...")
bert_sonuclari = batch_etiketle(bert_tr_pipe, metinler, BERT_TR_ETIKET_MAP)
bert_etiket    = [s[0] for s in bert_sonuclari]
bert_conf      = [s[1] for s in bert_sonuclari]

# 5d. Nihai karar
print("\nNihai etiketler hesaplanıyor...")
nihai = [
    nihai_etiket_belirle(
        kural_etiketi[i], kural_conf[i] or 0.0,
        xlm_etiket[i],    xlm_conf[i],
        bert_etiket[i],   bert_conf[i],
    )
    for i in range(len(metinler))
]
nihai_etiket  = [n[0] for n in nihai]
nihai_conf    = [n[1] for n in nihai]
nihai_kaynak  = [n[2] for n in nihai]

# ─────────────────────────────────────────────
# 6. SONUÇLARI DATAFRAME'E EKLE
# ─────────────────────────────────────────────
df["XLM_Roberta_Etiketi"]     = xlm_etiket
df["XLM_Roberta_Confidence"]  = xlm_conf
df["BERTurk_Etiketi"]         = bert_etiket
df["BERTurk_Confidence"]      = bert_conf
df["Kural_Etiketi"]           = kural_etiketi
df["Nihai_Etiket"]            = nihai_etiket
df["Nihai_Confidence"]        = nihai_conf
df["Karar_Kaynağı"]           = nihai_kaynak

# ─────────────────────────────────────────────
# 7. KAYDET
# ─────────────────────────────────────────────
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\n✅ Etiketlenmiş veri kaydedildi: {OUTPUT_FILE}")

# ─────────────────────────────────────────────
# 8. ÖZET İSTATİSTİK
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("ETİKET DAĞILIMI (Nihai):")
print(df["Nihai_Etiket"].value_counts().to_string())

print("\nKARAR KAYNAĞI DAĞILIMI:")
print(df["Karar_Kaynağı"].value_counts().to_string())

print(f"\nOrtalama confidence: {df['Nihai_Confidence'].mean():.3f}")
print(f"Düşük güven (<0.55) yorum sayısı: {(df['Nihai_Confidence'] < 0.55).sum()}")
print("="*50)

# Örnek çıktı
print("\nÖRNEK SATIRLAR:")
print(df[[TEXT_COL, "XLM_Roberta_Etiketi", "XLM_Roberta_Confidence",
          "BERTurk_Etiketi", "BERTurk_Confidence",
          "Nihai_Etiket", "Nihai_Confidence", "Karar_Kaynağı"]].head(10).to_string())

KeyboardInterrupt: 

In [3]:
import pandas as pd 
import numpy as np 
df = pd.read_csv('anaveri.csv', encoding='utf-8-sig')
df.head(1)

,Video Başlığı,Video Kaynağı,Yorum ID,Üst Yorum ID,Yorum Yapan Kişi,Yanıtlanan Kişi,Yorum Metni,Beğeni Sayısı,Tarih,Yanıt Mı?,XLM_Roberta_Etiketi,XLM_Roberta_Confidence,Kural_Etiketi
0,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgyJ0kV7TGCEACL_PDF4AaABAg,NaN,@Tugbayiilmaaz,NaN,AÇIK MUTFAK: ÇOCUKLUK TRAVMALARIM..,"5,9 B",1 yıl önce,Hayır,negative,0.8949,NaN


In [4]:
print(df['XLM_Roberta_Etiketi'].value_counts())

XLM_Roberta_Etiketi
positive    3678
negative    2148
neutral     1783
Name: count, dtype: int64


In [ ]:
print(df['Kural_Etiketi'].value_counts())

Kural_Etiketi
öneri/tavsiye    387
şikayet           44
Name: count, dtype: int64


In [6]:
# 2. XLM-RoBERTa Çıktılarını Türkçeleştirme (Veri Standardizasyonu)
# Eğer XLM modelin "positive, negative" gibi İngilizce çıktılar verdiyse,
# nihai sütunda karmaşa olmaması için önce onları Türkçe formatına çekiyoruz.
xlm_sozluk = {
    'positive': 'olumlu',
    'negative': 'olumsuz',
    'neutral': 'nötr'
}
df['XLM_Turkce'] = df['XLM_Roberta_Etiketi'].replace(xlm_sozluk)


# 3. NİHAİ ETİKET OLUŞTURMA (Ana Mantık)
# fillna() metodu şuna bakar: "Kural_Etiketi" sütunu NaN (boş) mu?
# Boşsa, o satırı git 'XLM_Turkce' sütunundaki değer ile doldur.
df['Nihai_Etiket'] = df['Kural_Etiketi'].fillna(df['XLM_Turkce'])


# 4. Sonuçları Kontrol Etme
print("Nihai Etiket Dağılımı:")
print(df['Nihai_Etiket'].value_counts())

# İşlemi biten temiz veriyi kaydet
df.to_csv('anaveri.csv', index=False, encoding='utf-8')
print("\nİşlem tamamlandı, yeni dosya kaydedildi.")

Nihai Etiket Dağılımı:
Nihai_Etiket
olumlu           3487
olumsuz          2020
nötr             1671
öneri/tavsiye     387
şikayet            44
Name: count, dtype: int64

İşlem tamamlandı, yeni dosya kaydedildi.


In [7]:
df.head(1)

,Video Başlığı,Video Kaynağı,Yorum ID,Üst Yorum ID,Yorum Yapan Kişi,Yanıtlanan Kişi,Yorum Metni,Beğeni Sayısı,Tarih,Yanıt Mı?,XLM_Roberta_Etiketi,XLM_Roberta_Confidence,Kural_Etiketi,XLM_Turkce,Nihai_Etiket
0,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgyJ0kV7TGCEACL_PDF4AaABAg,NaN,@Tugbayiilmaaz,NaN,AÇIK MUTFAK: ÇOCUKLUK TRAVMALARIM..,"5,9 B",1 yıl önce,Hayır,negative,0.8949,NaN,olumsuz,olumsuz
